# scRNA-seq Analysis Pipeline

This notebook documents the full single-cell RNA-seq data processing pipeline for the validation cohort, including quality control, doublet detection, lineage-specific re-clustering, cell type annotation, epithelial subclustering, and somatic variant-based single-cell subcloning analysis.

**Reproducibility note**: Stochastic steps (UMAP, Leiden clustering, Harmony batch correction) were computed during the original analysis and their outputs are stored in the intermediate `.h5ad` files referenced throughout this notebook. When the corresponding checkpoint directory already exists, the notebook loads the pre-computed results rather than re-running these steps, ensuring that cell type annotations and UMAP coordinates are identical to those reported in the manuscript.

In [1]:
import importlib.metadata
import sys

packages = [
    'scanpy', 'anndata', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'scipy',
    'harmonypy', 'scrublet',
]

print(f'Python: {sys.version}')
for pkg in packages:
    try:
        print(f'{pkg}: {importlib.metadata.version(pkg)}')
    except importlib.metadata.PackageNotFoundError:
        print(f'{pkg}: not found')

Python: 3.9.18 | packaged by conda-forge | (main, Dec 23 2023, 16:35:41) 
[Clang 16.0.6 ]
scanpy: 1.10.1
anndata: 0.10.8
pandas: 2.2.2
numpy: 1.26.4
matplotlib: 3.9.1
seaborn: 0.13.2
scipy: 1.13.1
harmonypy: 0.0.10
scrublet: not found


## 1. Import Libraries and Helper Functions

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import pandas as pd
import matplotlib.pyplot as plt
import math
from matplotlib import rcParams
def _figsize(x, y): rcParams['figure.figsize'] = (x, y)

import scanpy as sc
sc.set_figure_params(
    fontsize = 12,
    color_map = 'Reds'
)

def _tukey(vec):
    Q1 = vec.quantile(0.25)
    Q3 = vec.quantile(0.75)
    IQR = Q3 - Q1
    return (vec < Q1 - 1.5 * IQR) | (vec > Q3 + 1.5 * IQR)

# Library kit: Chromium Next GEM Single Cell 5p RNA library v2
# https://www.10xgenomics.com/support/single-cell-immune-profiling/documentation/steps/library-prep/chromium-single-cell-5-reagent-kits-user-guide-v-2-chemistry-dual-index
def _multipletRate(cell_number):
    if cell_number < 500:
        return 0.004
    elif cell_number < 1000:
        return 0.008
    elif cell_number < 2000:
        return 0.016
    elif cell_number < 3000:
        return 0.023
    elif cell_number < 4000:
        return 0.031
    elif cell_number < 5000:
        return 0.039
    elif cell_number < 6000:
        return 0.046
    elif cell_number < 7000:
        return 0.054
    elif cell_number < 8000:
        return 0.061
    elif cell_number < 9000:
        return 0.069
    elif cell_number < 10000:
        return 0.076
    else:
        print('warning: > 10000 cells')
        return 0.1

mypath = '/path/to/data'

# Helper: load 10x h5 files, apply QC metrics, and integrate samples
def _createAdata(sample_lst):
    adata_lst = {}
    for sample in sample_lst['sample_vec']:
        adata_lst[sample] = sc.read_10x_h5(f"{mypath}/data/h5/{sample}_filtered_feature_bc_matrix.h5")
        adata_lst[sample].var_names_make_unique(join = '.')
        adata_lst[sample].obs.index = sample + '__' + adata_lst[sample].obs.index
        if 'CellID_vec' in sample_lst:
            adata_lst[sample] = adata_lst[sample][adata_lst[sample].obs.index.isin(sample_lst['CellID_vec'])]

    adata = sc.concat(adata_lst, label = 'orig.ident')
    sc.pp.filter_cells(adata, min_genes = 200)
    sc.pp.filter_genes(adata, min_cells = 3)

    adata.obs['CellID'] = adata.obs.index
    adata.obs = pd.merge(
        adata.obs,
        metadata,
        how = 'left',
        on = 'orig.ident'
    ).set_index(adata.obs['CellID'])
    adata.obs.index.name = None

    if 'GAPDH' in adata.var.index:
        adata.var['mt'] = adata.var_names.str.startswith('MT-')
        adata.var['ribo'] = adata.var_names.str.startswith(('RPL', 'RPS'))
    elif 'Gapdh' in adata.var.index:
        adata.var['mt'] = adata.var_names.str.startswith('mt-')
        adata.var['ribo'] = adata.var_names.str.startswith(('Rpl', 'Rps'))
    else:
        print('percent_mt / percent_ribo need to be calculated manually')

    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars = ['mt', 'ribo'],
        percent_top = None,
        inplace = True,
        log1p = False
    )

    adata.obs.to_csv(f"./{progress_folder}/raw.csv")
    adata.write(f"./{progress_folder}/raw.h5ad")

    return adata

## 2. Quality Control (QC)

Initial QC pass: load all samples, compute per-cell QC metrics (number of genes, total counts, mitochondrial fraction, ribosomal fraction), perform dimensionality reduction, Leiden clustering, and assign broad cell type labels. Cells identified as Debris or Doublet are removed using Tukey outlier detection per cluster.

In [ ]:
progress_folder = '01_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

In [ ]:
sample_lst = {}
sample_lst['sample_vec'] = metadata['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

_figsize(8,8)
sc.pl.pca_variance_ratio(adata, n_pcs = 50)

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')

sc.tl.umap(adata, min_dist = 0.3, n_components = 2)

adata.obs.to_csv(f"./{progress_folder}/integrate.csv")
adata.write(f"./{progress_folder}/integrate.h5ad")

sc.pl.umap(adata, color = 'orig.ident')

gene_vec = ['PTPRC','PECAM1','DCN','RGS5','KRT8','MKI67']

sc.pl.umap(
    adata,
    color = ['n_genes', 'pct_counts_mt'] + gene_vec,
    legend_loc = 'on data',
    ncols = 4
)
resolution_vec = [
    0.3,0.4,0.5
]

for resolution in resolution_vec:
    sc.tl.leiden(
        adata,
        key_added = f"leiden_{resolution}",
        resolution = resolution
    )
    # Expects logarithmized data.
    sc.tl.rank_genes_groups(
        adata,
        layer = 'data',
        groupby = f"leiden_{resolution}",
        method = 'wilcoxon'
    )
    print(f"resolution: {resolution}")
    for i in adata.obs[f"leiden_{resolution}"].cat.categories:
        print(f"cluster_{i}: {','.join(sc.get.rank_genes_groups_df(adata, group=i).head(20)['names'].values)}")
    print('\n')

_figsize(8,8)
sc.pl.umap(adata, color = ['leiden_' + str(i) for i in resolution_vec], legend_loc = 'on data', ncols = 4)

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.3'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11,12,13
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO' # CILIATED
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['total_counts'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'adata_clusters', size = 0)

fig, axs = plt.subplots(
    math.ceil(len(adata.obs['adata_clusters'].cat.categories) / 4), 4,
    figsize = (16, math.ceil(len(adata.obs['adata_clusters'].cat.categories) / 4) * 4)
)

for i in adata.obs['adata_clusters'].cat.categories.to_list():
    sc.pl.scatter(
        adata[adata.obs['adata_clusters'] == i],
        x = 'n_genes',
        y = 'pct_counts_mt',
        size = 10,
        title = i,
        show = False,
        ax = axs[math.floor(int(i) / 4), int(i) % 4]
    )
    axs[math.floor(int(i) / 4), int(i) % 4].set_ylim(0, 100)
    axs[math.floor(int(i) / 4), int(i) % 4].set_xlim(0, 8000)

plt.tight_layout()

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'Fibroblast',
    '2':'Ciliated',
    '3':'Epithelial', # At this stage do not label as 'Debris'; defer to outlier removal
    '4':'SMC/Pericyte',
    '5':'Macrophage',
    '6':'T/NK',
    '7':'Endothelial',
    '8':'Debris',
    '9':'B/Plasma',
    '10':'Doublet',
    '11':'Mast',
    '12':'Epithelial',
    '13':'Epithelial',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'CAF',
    '2':'Epithelial',
    '3':'Epithelial',
    '4':'CAF',
    '5':'Immune',
    '6':'Immune',
    '7':'Vascular',
    '8':'Debris',
    '9':'Immune',
    '10':'Doublet',
    '11':'Immune',
    '12':'Epithelial',
    '13':'Epithelial',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

In [ ]:
_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['total_counts'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'adata_clusters', size = 0)

adata.obs['n_genes_outlier'] = adata.obs.groupby('adata_clusters')['n_genes'].apply(_tukey).reset_index(level = 0, drop = True)
adata.obs['counts_outlier'] = adata.obs.groupby('adata_clusters')['total_counts'].apply(_tukey).reset_index(level = 0, drop = True)
adata.obs['mt_outlier'] = adata.obs.groupby('adata_clusters')['pct_counts_mt'].apply(_tukey).reset_index(level = 0, drop = True)
adata.obs['ribo_outlier'] = adata.obs.groupby('adata_clusters')['pct_counts_ribo'].apply(_tukey).reset_index(level = 0, drop = True)

# Apply Tukey IQR-based outlier filter
adata = adata[
    (adata.obs['n_genes_outlier'] == False) &
    (adata.obs['counts_outlier'] == False) &
    (adata.obs['mt_outlier'] == False) &
    (adata.obs['ribo_outlier'] == False)
].copy()

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['total_counts'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'adata_clusters', size = 0)

adata.obs.to_csv(f"./{progress_folder}/final.csv")
adata.write(f"./{progress_folder}/final.h5ad")

In [ ]:
_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

## 3. Doublet Detection and Removal

Doublet detection using Scrublet. Each sample is scored independently using the expected doublet rate based on the 10x Genomics multiplet rate table. Cells manually identified as Debris or Doublet in the QC step are removed first, then Scrublet-predicted doublets are removed.

In [ ]:
progress_folder = '02_doublet'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./01_qc/final.csv', index_col = 0)
print(meta_df['cell_category'].unique())

In [ ]:
# Exclude Debris at this step; Doublets are kept until Scrublet scoring
meta_df = meta_df[~meta_df['cell_type'].isin(['Debris'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata_lst = {}
for sample in sample_lst['sample_vec']:
    # Read h5 file
    adata_lst[sample] = sc.read_10x_h5(f"{mypath}/data/h5/{sample}_filtered_feature_bc_matrix.h5")
    adata_lst[sample].var_names_make_unique()
    adata_lst[sample].obs.index = sample + '__' + adata_lst[sample].obs.index
    _cell_number = adata_lst[sample].n_obs
    _rate = _multipletRate(_cell_number)
    adata_lst[sample] = adata_lst[sample][adata_lst[sample].obs.index.isin(sample_lst['CellID_vec'])]
    print(sample)
    print(f"before filter: {_cell_number} cells, after filter: {adata_lst[sample].n_obs} cells, expected_doublet_rate: {_rate}")
    sc.pp.scrublet(adata_lst[sample], expected_doublet_rate = _rate) # Doublet prediction by Scrublet

doublet_df = sc.concat(adata_lst).obs
doublet_df['CellID'] = doublet_df.index

adata = adata[~adata.obs['cell_type'].isin(['Debris','Doublet'])] # Remove manually identified Doublets from 01_qc
adata.obs = pd.merge(adata.obs, doublet_df, on = 'CellID', how = 'left').set_index('CellID', drop = False)
adata.obs.index.name = None

_figsize(8,8)
sc.pl.umap(adata, color = 'predicted_doublet', size = 20)

adata.obs.to_csv(f"./{progress_folder}/doublet.csv") # Scrublet doublet predictions not yet removed at this point
adata.write(f"./{progress_folder}/doublet.h5ad")

## 4. Pre-clustering

After removing Scrublet-predicted doublets, perform a second round of integration and clustering to assign broad cell category labels (Epithelial, CAF, Vascular, Immune) for downstream lineage-specific re-clustering.

In [ ]:
progress_folder = '03_preclustering'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./02_doublet/doublet.csv', index_col = 0)
print(meta_df['predicted_doublet'].unique())

In [ ]:
meta_df = meta_df[~meta_df['predicted_doublet']]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

_figsize(8,8)
sc.pl.pca_variance_ratio(adata, n_pcs = 50)

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')

sc.tl.umap(adata, min_dist = 0.3, n_components = 2)

adata.obs.to_csv(f"./{progress_folder}/integrate.csv")
adata.write(f"./{progress_folder}/integrate.h5ad")

sc.pl.umap(adata, color = 'orig.ident')

gene_vec = ['PTPRC','PECAM1','DCN','RGS5','KRT8','MKI67']

sc.pl.umap(
    adata,
    color = ['n_genes', 'pct_counts_mt'] + gene_vec,
    legend_loc = 'on data',
    ncols = 4
)

resolution_vec = [
    0.3,0.4,0.5
]

for resolution in resolution_vec:
    sc.tl.leiden(
        adata,
        key_added = f"leiden_{resolution}",
        resolution = resolution
    )
    sc.tl.rank_genes_groups(
        adata,
        layer = 'data',
        groupby = f"leiden_{resolution}",
        method = 'wilcoxon'
    )
    print(f"resolution: {resolution}")
    for i in adata.obs[f"leiden_{resolution}"].cat.categories:
        print(f"cluster_{i}: {','.join(sc.get.rank_genes_groups_df(adata, group=i).head(20)['names'].values)}")
    print('\n')

_figsize(8,8)
sc.pl.umap(adata, color = ['leiden_' + str(i) for i in resolution_vec], legend_loc = 'on data', ncols = 4)

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.3'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO' # CILIATED
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['total_counts'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'adata_clusters', size = 0)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'adata_clusters', size = 0)

fig, axs = plt.subplots(
    math.ceil(len(adata.obs['adata_clusters'].cat.categories) / 4), 4,
    figsize = (16, math.ceil(len(adata.obs['adata_clusters'].cat.categories) / 4) * 4)
)

for i in adata.obs['adata_clusters'].cat.categories.to_list():
    sc.pl.scatter(
        adata[adata.obs['adata_clusters'] == i],
        x = 'n_genes',
        y = 'pct_counts_mt',
        size = 10,
        title = i,
        show = False,
        ax = axs[math.floor(int(i) / 4), int(i) % 4]
    )
    axs[math.floor(int(i) / 4), int(i) % 4].set_ylim(0, 100)
    axs[math.floor(int(i) / 4), int(i) % 4].set_xlim(0, 8000)

plt.tight_layout()

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'Fibroblast',
    '2':'Ciliated',
    '3':'Epithelial',
    '4':'SMC/Pericyte',
    '5':'Macrophage',
    '6':'T/NK',
    '7':'Endothelial',
    '8':'Macrophage',
    '9':'Mast',
    '10':'Epithelial',
    '11':'Epithelial',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'CAF',
    '2':'Epithelial',
    '3':'Epithelial',
    '4':'CAF',
    '5':'Immune',
    '6':'Immune',
    '7':'Vascular',
    '8':'Immune',
    '9':'Immune',
    '10':'Epithelial',
    '11':'Epithelial',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

## 5. Lineage-specific Re-clustering and Annotation

Each major cell lineage (mesenchymal/CAF, CAF, vascular endothelial, T/NK, immune [B/Plasma/Macrophage/Mast], epithelial, ciliated) is extracted based on the pre-clustering labels and re-clustered at higher resolution for refined annotation.

### 5.1 Mesenchymal Pre-clustering (CAF + Vascular)

In [ ]:
progress_folder = '04_mesenchymal_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./03_preclustering/cluster.csv', index_col = 0)
print(meta_df['cell_category'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_category'].isin(['Vascular', 'CAF'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

_figsize(8,8)
sc.pl.pca_variance_ratio(adata, n_pcs = 50)

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')

sc.tl.umap(adata, min_dist = 0.3, n_components = 2)

adata.obs.to_csv(f"./{progress_folder}/integrate.csv")
adata.write(f"./{progress_folder}/integrate.h5ad")

sc.pl.umap(adata, color = 'orig.ident')

gene_vec = ['PTPRC','PECAM1','DCN','RGS5','KRT8','MKI67']

sc.pl.umap(
    adata,
    color = ['n_genes', 'pct_counts_mt'] + gene_vec,
    legend_loc = 'on data',
    ncols = 4
)

resolution_vec = [
    0.4
]

for resolution in resolution_vec:
    sc.tl.leiden(
        adata,
        key_added = f"leiden_{resolution}",
        resolution = resolution
    )
    sc.tl.rank_genes_groups(
        adata,
        layer = 'data',
        groupby = f"leiden_{resolution}",
        method = 'wilcoxon'
    )
    print(f"resolution: {resolution}")
    for i in adata.obs[f"leiden_{resolution}"].cat.categories:
        print(f"cluster_{i}: {','.join(sc.get.rank_genes_groups_df(adata, group=i).head(20)['names'].values)}")
    print('\n')

_figsize(8,8)
sc.pl.umap(adata, color = ['leiden_' + str(i) for i in resolution_vec], legend_loc = 'on data', ncols = 4)

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.4'

level = [
    0,1,2,3,4,5,6,7,8,9,10
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', 'TAGLN', # VSMC, myCAF
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO', # CILIATED
'CD74','HLA-DRB1', # iCAF
'PROX1','PDPN', # Lymphatic endothelial
'RAMP2','EMCN', # Capillary
'NR2F2', # venous
 ]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Fibroblast',
    '1':'SMC/Pericyte',
    '2':'Endothelial',
    '3':'Fibroblast',
    '4':'Fibroblast',
    '5':'Fibroblast',
    '6':'Fibroblast',
    '7':'Doublet',
    '8':'Fibroblast',
    '9':'Fibroblast',
    '10':'Doublet',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'CAF',
    '1':'CAF',
    '2':'Vascular',
    '3':'CAF',
    '4':'CAF',
    '5':'CAF',
    '6':'CAF',
    '7':'Doublet',
    '8':'CAF',
    '9':'CAF',
    '10':'Doublet',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.2 CAF Sub-clustering

In [ ]:
progress_folder = '05_caf_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./04_mesenchymal_qc/cluster.csv', index_col = 0)
print(meta_df['cell_category'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_category'].isin(['CAF'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.4', resolution = 0.4)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.4', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.4'

level = [
    0,1,2,3,4,5,6,7
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO', # CILIATED
'CD74','HLA-DRB1', # iCAF
'DIO2', 'ALDH1A1', # secretory-like CAF
'TAGLN', 'MYH9', # myCAF
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'iCAF',
    '1':'myCAF',
    '2':'ECM-CAF',
    '3':'SMC-CAF',
    '4':'KI67-fibro',
    '5':'ECM-CAF',
    '6':'Doublet',
    '7':'ECM-CAF',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'CAF',
    '1':'CAF',
    '2':'CAF',
    '3':'CAF',
    '4':'CAF',
    '5':'CAF',
    '6':'Doublet',
    '7':'CAF',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category','cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.3 Vascular Sub-clustering

In [ ]:
progress_folder = '06_vascular_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./04_mesenchymal_qc/cluster.csv', index_col = 0)
print(meta_df['cell_category'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_category'].isin(['Vascular'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.4', resolution = 0.4)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.4', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.4'

level = [
    0,1,2,3,4,5,6,7
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'GNLY', # NK
'JCHAIN', # PLASMA
'CD68', # MACROPHAGE
'EPCAM',
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO', # CILIATED
'PROX1','PDPN', # Lymphatic endothelial
'RAMP2','EMCN', # Capillary
'NR2F2', # venous
'ACKR1','CD74', # antigen-presenting endothelial
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Infla-Endo',
    '1':'Endothelial',
    '2':'Infla-Endo', # Clusters 0 and 2 are similar; grouped together
    '3':'Lymph-Endo',
    '4':'Debris',
    '5':'Doublet',
    '6':'KI67-Endo',
    '7':'Doublet',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Vascular',
    '1':'Vascular',
    '2':'Vascular',
    '3':'Vascular',
    '4':'Debris',
    '5':'Doublet',
    '6':'Vascular',
    '7':'Doublet',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.4 T/NK Sub-clustering

In [ ]:
progress_folder = '07_nkt_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./03_preclustering/cluster.csv', index_col = 0)
print(meta_df['cell_type'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_type'].isin(['T/NK'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.6', resolution = 0.6)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.6', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.6'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'FOXP3', 'CTLA4', # Treg
'IL7R',
'TCF7',
'XCL1', 'XCL2',
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Treg',
    '1':'Effector-CD8',
    '2':'em-CD8',
    '3':'Naive-CD4',
    '4':'Debris',
    '5':'KI67-T',
    '6':'NK/innate',
    '7':'Active-CD4',
    '8':'KI67-T',
    '9':'Doublet',
    '10':'Doublet',
    '11':'Doublet',
    '12':'Doublet',
    '13':'ILC2-like',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Immune',
    '1':'Immune',
    '2':'Immune',
    '3':'Immune',
    '4':'Debris',
    '5':'Immune',
    '6':'Immune',
    '7':'Immune',
    '8':'Immune',
    '9':'Doublet',
    '10':'Doublet',
    '11':'Doublet',
    '12':'Doublet',
    '13':'Immune',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.5 Immune Sub-clustering (B/Plasma, Macrophage, Mast)

In [ ]:
progress_folder = '08_immune_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./03_preclustering/cluster.csv', index_col = 0)
print(meta_df['cell_type'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_type'].isin(['B/Plasma', 'Macrophage', 'Mast'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.4', resolution = 0.4)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.4', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.4'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'IL1B','LYZ', 'CD68', 'APOE', # MACROPHAGE
'CD1C','CD14','FCGR2A',
'IRF8','CLEC9A',
'CCR7','LAMP3',
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Plasma',
    '1':'M2-Mac',
    '2':'cDC2-like',
    '3':'Infla-Mono',
    '4':'Debris',
    '5':'Mast',
    '6':'KI67-Mac',
    '7':'Doublet',
    '8':'Doublet',
    '9':'Doublet',
    '10':'cDC1-like',
    '11':'migDC',
    '12':'B',
    '13':'Plasma',
    '14':'Plasma',
    '15':'pDC',
    '16':'Plasma',
    '17':'Doublet',
    '18':'Plasma',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Immune',
    '1':'Immune',
    '2':'Immune',
    '3':'Immune',
    '4':'Debris',
    '5':'Immune',
    '6':'Immune',
    '7':'Doublet',
    '8':'Doublet',
    '9':'Doublet',
    '10':'Immune',
    '11':'Immune',
    '12':'Immune',
    '13':'Immune',
    '14':'Immune',
    '15':'Immune',
    '16':'Immune',
    '17':'Doublet',
    '18':'Immune',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.6 Epithelial Sub-clustering

In [ ]:
progress_folder = '09_epi_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./03_preclustering/cluster.csv', index_col = 0)
print(meta_df['cell_type'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_type'].isin(['Epithelial'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.6', resolution = 0.6)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.6', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.6'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11,12,13
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO', # CILIATED
'SOX9'
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'Epithelial',
    '2':'Debris',
    '3':'Epithelial',
    '4':'Epithelial',
    '5':'Debris',
    '6':'Epithelial',
    '7':'Debris',
    '8':'Epithelial',
    '9':'Doublet',
    '10':'Doublet',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

### 5.7 Ciliated Cell Sub-clustering

In [ ]:
progress_folder = '10_ciliated_qc'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

meta_df = pd.read_csv('./03_preclustering/cluster.csv', index_col = 0)
print(meta_df['cell_type'].unique())

In [ ]:
meta_df = meta_df[meta_df['cell_type'].isin(['Ciliated'])]

sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')
sc.tl.umap(adata, min_dist = 0.3, n_components = 2)
sc.tl.leiden(adata, key_added = 'leiden_0.5', resolution = 0.5)
sc.tl.rank_genes_groups(adata, layer = 'data', groupby = 'leiden_0.5', method = 'wilcoxon')

adata.obs.to_csv(f"./{progress_folder}/resolution.csv")
adata.write(f"./{progress_folder}/resolution.h5ad")

In [ ]:
resolution = 'leiden_0.5'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', # NK
'LYZ', 'CD68', # MACROPHAGE
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO' # CILIATED
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'Epithelial',
    '2':'Epithelial',
    '3':'Epithelial',
    '4':'Debris', # Borderline; assigned as Debris
    '5':'Epithelial',
    '6':'Epithelial',
    '7':'Epithelial',
    '8':'Epithelial',
    '9':'Doublet',
    '10':'Doublet',
})

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/cluster.csv")
adata.write(f"./{progress_folder}/cluster.h5ad")

## 6. Final Integrated Clustering

Combine cell type annotations from all lineage-specific re-clustering steps and reconstruct an integrated AnnData object containing all high-quality cells.

In [ ]:
progress_folder = '11_clustering'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

In [ ]:
caf_df = pd.read_csv('./05_caf_qc/cluster.csv', index_col = 0)
vascular_df = pd.read_csv('./06_vascular_qc/cluster.csv', index_col = 0)
nkt_df = pd.read_csv('./07_nkt_qc/cluster.csv', index_col = 0)
immune_df = pd.read_csv('./08_immune_qc/cluster.csv', index_col = 0)
epi_df = pd.read_csv('./09_epi_qc/cluster.csv', index_col = 0)
ciliated_df = pd.read_csv('./10_ciliated_qc/cluster.csv', index_col = 0)

meta_df = pd.concat(
    [caf_df, vascular_df, nkt_df, immune_df, epi_df, ciliated_df], axis = 0
)

print(meta_df['cell_category'].unique())

In [ ]:
meta_df = meta_df[~meta_df['cell_category'].isin(['Doublet', 'Debris'])]

sample_lst = {}
sample_lst['sample_vec'] = metadata['orig.ident']
sample_lst['CellID_vec']  = meta_df['CellID']

In [ ]:
adata = _createAdata(sample_lst)

_figsize(6,1)
sc.pl.violin(adata, ['n_genes'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['total_counts'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_mt'], groupby = 'orig.ident', size = 0, rotation = 90)
sc.pl.violin(adata, ['pct_counts_ribo'], groupby = 'orig.ident', size = 0, rotation = 90)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e6)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key = 'orig.ident')
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata, n_comps = 50)
adata.layers['scale_data'] = adata.X.copy()
adata.X = adata.layers['data']

_figsize(8,8)
sc.pl.pca_variance_ratio(adata, n_pcs = 50)

sc.external.pp.harmony_integrate(
    adata,
    key = 'orig.ident',
    basis = 'X_pca',
    adjusted_basis = 'X_harmony',
    max_iter_harmony = 50
)

sc.pp.neighbors(adata, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')

sc.tl.umap(adata, min_dist = 0.3, n_components = 2)

adata.obs.to_csv(f"./{progress_folder}/integrate.csv")
adata.write(f"./{progress_folder}/integrate.h5ad")

sc.pl.umap(adata, color = 'orig.ident')

In [ ]:
resolution = 'leiden_0.1'

level = [
    0,1,2,3,4,5,6,7,8,9
]

adata.obs['adata_clusters'] = pd.Categorical(
    adata.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'PTPRC', # IMMUNE
'CD3D', 'CD3E', # T
'GNLY', 'KLRD1', # NK
'IL7R',
'CD8A',
'CTLA4', 'FOXP3',
'XCL1', 'XCL2',
'CD79A', 'MS4A1', # B
'JCHAIN', # PLASMA
'LYZ', 'CD68', # MACROPHAGE
'TPSAB1', 'TPSB2', # MAST
'PECAM1', 'VWF', # ENDOTHELIAL
'RGS5', 'PDGFRB', # PERICYTE
'ACTA2', 'MYH11', # VSMC
'DCN', 'PDGFRA', # FIBROBLAST
'KRT8', 'KRT18', # EPITHELIAL
'FOXJ1', 'PIFO' # CILIATED
]

sc.pl.dotplot(adata, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
adata.obs['cell_type'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'Fibroblast',
    '2':'Ciliated',
    '3':'SMC/Pericyte',
    '4':'Macrophage',
    '5':'T/NK',
    '6':'Endothelial',
    '7':'Macrophage', # Broadly grouped; includes B/Plasma
    '8':'Mast',
})

adata.obs['cell_category'] = adata.obs['adata_clusters'].map({
    '0':'Epithelial',
    '1':'CAF',
    '2':'Epithelial',
    '3':'CAF',
    '4':'Immune',
    '5':'Immune',
    '6':'Vascular',
    '7':'Immune',
    '8':'Immune',
})

In [ ]:
adata = adata[~adata.obs['cell_type'].isin(['Doublet', 'Debris'])]

_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

adata.obs.to_csv(f"./{progress_folder}/final.csv")
adata.write(f"./{progress_folder}/final.h5ad")

In [ ]:
adata.obs.groupby(['cell_type', 'orig.ident'])['cell_type'].agg('count').unstack()

## 7. Epithelial Subclustering

Extract epithelial cells from the final integrated object and perform UMAP re-embedding and Leiden clustering at higher resolution to identify epithelial subtypes including the SOX9+ basal-like population (S9), common cancer progenitor (L5_S9_DP), proliferating cells (KI67), luminal cells, and patient-specific clusters.

In [ ]:
progress_folder = '11_subclustering'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

In [ ]:
meta_df = pd.read_csv('./11_clustering/final.csv', index_col = 0)
print(meta_df['cell_category'].unique())

In [ ]:
# Load integrated data and extract epithelial cells
adata = sc.read_h5ad('./11_clustering/final.h5ad')
# Extract epithelial cells
epithelial_cells = adata[adata.obs['cell_category'] == 'Epithelial'].copy()

In [ ]:
_figsize(8,8)
sc.pl.umap(adata, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

In [ ]:
_figsize(8,8)
sc.pl.umap(epithelial_cells, color = ['cell_category', 'cell_type'], legend_loc = 'on data')

In [ ]:
sc.pp.neighbors(epithelial_cells, n_neighbors = 20, n_pcs = 50, use_rep = 'X_harmony')

sc.tl.umap(epithelial_cells, min_dist = 0.3, n_components = 2)

sc.pl.umap(epithelial_cells, color = 'orig.ident')

gene_vec = ['PTPRC','PECAM1','DCN','RGS5','KRT8','MKI67', 'LGR5', 'CDH2', 'MSLN','SAA1']

sc.pl.umap(
    epithelial_cells,
    color = ['n_genes', 'pct_counts_mt'] + gene_vec,
    legend_loc = 'on data',
    ncols = 4
)

resolution_vec = [
    0.6,0.7,0.8,0.9,1.0,1.1,1.2
]

for resolution in resolution_vec:
    sc.tl.leiden(
        epithelial_cells,
        key_added = f"leiden_{resolution}",
        resolution = resolution
    )
    sc.tl.rank_genes_groups(
        epithelial_cells,
        layer = 'data',
        groupby = f"leiden_{resolution}",
        method = 'wilcoxon'
    )
    print(f"resolution: {resolution}")
    for i in epithelial_cells.obs[f"leiden_{resolution}"].cat.categories:
        print(f"cluster_{i}: {','.join(sc.get.rank_genes_groups_df(epithelial_cells, group=i).head(20)['names'].values)}")
    print('\n')

_figsize(8,8)
sc.pl.umap(epithelial_cells, color = ['leiden_' + str(i) for i in resolution_vec], legend_loc = 'on data', ncols = 4)

In [ ]:
resolution = 'leiden_0.8'

level = [
    0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
]

epithelial_cells.obs['adata_clusters'] = pd.Categorical(
    epithelial_cells.obs[resolution],
    categories = [str(i) for i in level],
    ordered = True
)

gene_vec = [
'n_genes', 'pct_counts_mt', 'MKI67',
'KRT8', 'KRT18', # EPITHELIAL
'ESR1',
'SOX9', 'LGR5', 'CDH2', 'MMP7',
'SLC26A7', 'LEFTY1', 'PTGS1', 'MSLN', 'WNT7A', # Luminal
'SCGB2A1','SCGB2A2','KLF5', # glandular
'PAEP', 'CXCL14', 'DPP4', 'GPX3', # secretory
'MUC12', 'CDC20B', 'CCNO', 'HES6', # pre-ciliated
'FOXJ1', 'PIFO', # CILIATED
'NOTUM', 'DKK1', 'AXIN2', # WNT pathway
]

sc.pl.dotplot(epithelial_cells, gene_vec, groupby = 'adata_clusters', standard_scale = 'var', swap_axes = True)

In [ ]:
epithelial_cells.obs['cell_type'] = epithelial_cells.obs['adata_clusters'].map({
    '0':'L5_S9_DP',
    '1':'L5_S9_DP',
    '2':'Lumenal',
    '3':'Ciliated',
    '4':'Ciliated',
    '5':'Ciliated',
    '6':'KI67',
    '7':'S9',
    '8':'15T2',
    '9':'Preciliated',
    '10':'15T2',
    '11':'23T2',
    '12':'S9', # AEH0018-specific
    '13':'15T1',
    '14':'18T',
    '15':'23T2',
    '16':'46T2',
    '17':'15T2',
})

_figsize(8,8)
sc.pl.umap(epithelial_cells, color = ['cell_type'], legend_loc = 'on data')

In [ ]:
pd.crosstab(epithelial_cells.obs['orig.ident'], epithelial_cells.obs['cell_type'])

In [ ]:
epithelial_cells = epithelial_cells[~epithelial_cells.obs['cell_type'].isin(['Doublet', 'Debris'])]

epithelial_cells.obs.to_csv(f"./{progress_folder}/epithelial-cluster.csv")
epithelial_cells.write(f"./{progress_folder}/epithelial-cluster.h5ad")

In [ ]:
gene_vec = ['PTPRC','PECAM1','DCN','RGS5','KRT8','MKI67', 'LGR5', 'CDH2', 'SOX9', 'NOTUM']

sc.pl.umap(
    epithelial_cells,
    color = ['n_genes', 'pct_counts_mt'] + gene_vec,
    legend_loc = 'on data',
    ncols = 4
)

### DEG: SOX9 basal-like (S9) vs common cancer progenitor (L5_S9_DP)

In [ ]:
# Compare S9 vs L5_S9_DP populations
cluster1 = 'S9'
cluster2 = 'L5_S9_DP'

# Run DEG analysis
sc.tl.rank_genes_groups(epithelial_cells, groupby='cell_type', groups=[cluster1], reference=cluster2, method='wilcoxon')

# Display top DEGs
sc.pl.rank_genes_groups(epithelial_cells, n_genes=50, sharey=False)

## 8. Somatic Variant-based Single-cell Subcloning (cellSNP)

Single-cell subclone analysis using somatic variant information from cellSNP output. For each patient, we identify single-cell-level variant allele frequencies (VAF) across informative SNP loci, perform hierarchical clustering of epithelial cells, and assign subclone labels.

In [ ]:
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
def _figsize(x, y): rcParams['figure.figsize'] = (x, y)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times'

import scipy.io as sio
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster


mypath = '/path/to/data'

def _readCellsnp(sample_vec):

    ad_df = pd.DataFrame()
    for sample in sample_vec:
        with open(f"{mypath}/cellsnp/{sample}/cellSNP.base.vcf") as f:
            for i, line in enumerate(f):
                if line.startswith('#CHROM'):
                    header_num = i
                    break
        vcf_df = pd.read_csv(f"{mypath}/cellsnp/{sample}/cellSNP.base.vcf", sep = '\t', header = header_num)
        alt_vec = vcf_df['#CHROM'] + '_' + vcf_df['POS'].astype(str) + vcf_df['REF'] + '>' + vcf_df['ALT']
        cell_vec = pd.read_csv(f"{mypath}/cellsnp/{sample}/cellSNP.samples.tsv", header = None)[0].values
        df = pd.DataFrame(sio.mmread(f"{mypath}/cellsnp/{sample}/cellSNP.tag.AD.mtx").todense().T)
        df.index = sample + '__' + cell_vec
        df.columns = alt_vec
        ad_df = pd.concat([ad_df, df], axis = 0)

    dp_df = pd.DataFrame()
    for sample in sample_vec:
        with open(f"{mypath}/cellsnp/{sample}/cellSNP.base.vcf") as f:
            for i, line in enumerate(f):
                if line.startswith('#CHROM'):
                    header_num = i
                    break
        vcf_df = pd.read_csv(f"{mypath}/cellsnp/{sample}/cellSNP.base.vcf", sep = '\t', header = header_num)
        alt_vec = vcf_df['#CHROM'] + '_' + vcf_df['POS'].astype(str) + vcf_df['REF'] + '>' + vcf_df['ALT']
        cell_vec = pd.read_csv(f"{mypath}/cellsnp/{sample}/cellSNP.samples.tsv", header = None)[0].values
        df = pd.DataFrame((
            sio.mmread(f"{mypath}/cellsnp/{sample}/cellSNP.tag.DP.mtx") +
            sio.mmread(f"{mypath}/cellsnp/{sample}/cellSNP.tag.OTH.mtx")
        ).todense().T)
        df.index = sample + '__' + cell_vec
        df.columns = alt_vec
        dp_df = pd.concat([dp_df, df], axis = 0)

    merge_df = (ad_df > 0) * 1 + (dp_df > 0) * 1

    mut_sum_vec = (ad_df > 0).sum(axis = 0).sort_values(ascending = False)
    include_vec = mut_sum_vec.index[mut_sum_vec > 0]
    ad_df = ad_df.loc[:, include_vec]
    ad_df.insert(0, 'CellID', ad_df.index)
    ad_df = pd.merge(meta_df[['CellID', 'cell_category', 'tumor_cell']], ad_df, on = 'CellID', how = 'inner').set_index('CellID', drop = False)
    ad_df.index.name = None
    merge_df = merge_df.loc[:, include_vec]
    merge_df.insert(0, 'CellID', merge_df.index)
    merge_df = pd.merge(meta_df[['CellID', 'cell_category', 'tumor_cell']], merge_df, on = 'CellID', how = 'inner').set_index('CellID', drop = False)
    merge_df.index.name = None

    # Create copies to resolve dataframe fragmentation
    ad_df = ad_df.copy()
    merge_df = merge_df.copy()

    return ad_df, merge_df

def _snpInVcf(sample):
    sample_number = sample.replace('AEH', '')
    vcf_vec = [filename.replace('AEH-', 'AEH')
                                     .replace('-PB-filtered-snpeff.vcf', '')
                                     .replace('-T', '_T')
                                     .replace('-H', '_H')
                                     .replace('-N', '_N')
                      for filename in sorted(os.listdir(f'{mypath}/Mutect2VCF'))]
    vcf_vec = [i for i in vcf_vec if i.startswith(sample)]

    ad_lst = {}

    for vcf in vcf_vec:
        file_path = f"{mypath}/Mutect2VCF/AEH-{sample_number}-{vcf[8:]}-PB-filtered-snpeff.vcf"
        with open(file_path) as f:
            for i, line in enumerate(f):
                if line.startswith('#CHROM'):
                    header_num = i
                    break
        _df = pd.read_csv(file_path, sep = '\t', header = header_num)
        ad_lst[vcf.replace('-', '_').replace('_', '', 1)] = _df['#CHROM'] + '_' + _df['POS'].astype(str) + _df['REF'] + '>' + _df['ALT']

    df = pd.DataFrame()
    for i in ad_lst:
        df[i] = ad_df.columns[ad_df.columns.str.startswith('chr')].isin(ad_lst[i])
    df.index = ad_df.columns[ad_df.columns.str.startswith('chr')]

    return(df)

In [ ]:
progress_folder = '13_snp'

if os.path.exists(progress_folder):
    print('!!folder exists')
else:
    os.makedirs(progress_folder)

In [ ]:
metadata = pd.read_csv('./data/metadata.tsv', sep = '\t')
print(metadata)

In [ ]:
meta_df = pd.read_csv('./12_infercnv/infercnv.csv', index_col = 0)
print(meta_df['cell_category'].unique())

### Patient AEH0018

In [ ]:
id = 'AEH0018'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [3]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, [0,1]]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'chr1_204411583G>A',
        'SNP_2':'chrM_1273G>A',
        'SNP_3':'chrM_1273G>A',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0023

In [ ]:
id = 'AEH0023'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, 3:20]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'SNP_1',
        'SNP_2':'SNP_2',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0026

In [ ]:
id = 'AEH0026'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, :30]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'SNP_1',
        'SNP_2':'SNP_2',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0032

In [ ]:
id = 'AEH0032'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, 1:20]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'SNP_1',
        'SNP_2':'SNP_2',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0034

In [ ]:
id = 'AEH0034'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, :20]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'SNP_1',
        'SNP_2':'SNP_2',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0038

In [ ]:
id = 'AEH0038'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, [2,10]]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'None',
        'SNP_2':'chr11_66593660A>C',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Patient AEH0046

In [ ]:
id = 'AEH0046'

sample_lst = {}
sample_lst['sample_vec'] = metadata[metadata['orig.ident'].str.startswith(id)]['orig.ident']
print(sample_lst['sample_vec'])

In [ ]:
ad_df, merge_df = _readCellsnp(sample_lst['sample_vec'])

In [ ]:
snp_df = merge_df.sort_values('cell_category')

col_colors = snp_df['cell_category'].map({
    'Immune': 'green',
    'Vascular':'blue',
    'CAF':'orange',
    'Epithelial':'red'
})

ax = sns.clustermap(
    snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
    row_cluster = False,
    col_cluster = False,
    col_colors = col_colors,
    colors_ratio = 0.02,
    cmap = ['whitesmoke', 'darkgrey', 'red'],
    cbar_pos = None,
    xticklabels = False,
    yticklabels = True,
    figsize = (10,10))

ax.ax_row_dendrogram.remove()
ax.ax_col_dendrogram.remove()

In [ ]:
# Hierarchical clustering of epithelial cells by variant allele detection profile
cluster_num_vec = [2]

for i in cluster_num_vec:
    print(f"cluster_num: {i}")
    snp_df = ad_df[ad_df['cell_category'] == 'Epithelial'].loc[:, ad_df.columns.str.startswith('chr')].iloc[:, 3:20]
    distance_matrix = linkage(snp_df > 0, method = 'ward')
    print(snp_df.shape)

    snp_df = merge_df[merge_df['cell_category'] == 'Epithelial'].iloc[:, :15]
    snp_df.insert(0, 'snp_cluster', ['SNP_' + str(i) for i in fcluster(distance_matrix, t = i, criterion = 'maxclust')])
    snp_df = snp_df.sort_values('snp_cluster')

    color_fcluster = snp_df['snp_cluster'].map({
        'SNP_1': 'blue',
        'SNP_2': 'red',
        'SNP_3': 'green',
    })

    ax = sns.clustermap(
        snp_df.loc[:, snp_df.columns.str.startswith('chr')].T,
        row_cluster = False,
        col_cluster = False,
        col_colors = color_fcluster,
        colors_ratio = 0.02,
        cmap = ['whitesmoke', 'darkgrey', 'red'],
        cbar_pos = None,
        xticklabels = False,
        yticklabels = True,
        figsize = (10,10))

    ax.ax_row_dendrogram.remove()
    ax.ax_col_dendrogram.remove()

In [ ]:
mut_df = _snpInVcf(id)
mut_df.head()

In [ ]:
snp_df.insert(
    0,
    'SNP',
    snp_df['snp_cluster'].map({
        'SNP_1':'chr17_7307685A>G',
        'SNP_2':'chr9_110256458G>A',
    }
    )
)

snp_df.to_csv(f"./{progress_folder}/{id}_snp.csv")

### Final Data Aggregation

In [ ]:
# Load all lineage-specific subclustered AnnData objects
epithelial_cells = sc.read_h5ad('./11_subclustering/epithelial-cluster.h5ad')
caf_cells = sc.read_h5ad('./05_caf_qc/cluster.h5ad')
immune_cells = sc.read_h5ad('./08_immune_qc/cluster.h5ad')
nkt_cells = sc.read_h5ad('./07_nkt_qc/cluster.h5ad')
vascular_cells = sc.read_h5ad('./06_vascular_qc/cluster.h5ad')